In [1]:
%%configure -f
{
  "conf": {
    "spark.master": "yarn",
    "spark.speculation": "false",

    "spark.driver.cores": "8",
    "spark.driver.memory": "384g",
    "spark.driver.memoryOverhead": "96g",

    "spark.yarn.am.cores": "6",
    "spark.yarn.am.memory": "384g",
    "spark.yarn.am.memoryOverhead": "96g",

    "spark.driver.maxResultSize": "20g",

    "spark.executor.cores": "4",
    "spark.executor.memory": "18g",
    "spark.executor.memoryOverhead": "8g",

    "spark.dynamicAllocation.enabled": "true",
    "spark.dynamicAllocation.minExecutors": "189",
    "spark.dynamicAllocation.initialExecutors": "189",
    "spark.dynamicAllocation.maxExecutors": "1000",
    "spark.dynamicAllocation.executorIdleTimeout": "60s",
    "spark.dynamicAllocation.cachedExecutorIdleTimeout": "300s",
    "spark.dynamicAllocation.schedulerBacklogTimeout": "1s",
    "spark.dynamicAllocation.sustainedSchedulerBacklogTimeout": "1s",
    "spark.dynamicAllocation.executorAllocationRatio": "1.0",

    "spark.locality.wait": "1s",

    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "false",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "268435456",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.localShuffleReader.enabled": "true",
    "spark.sql.shuffle.partitions": "1024",
    "spark.default.parallelism": "1024",

    "spark.network.timeout": "800s",
    "spark.executor.heartbeatInterval": "60s",
    "spark.kryoserializer.buffer.max": "1g",
    "spark.rpc.message.maxSize": "1024",

    "spark.hadoop.fs.s3a.aws.credentials.provider": "com.amazonaws.auth.DefaultAWSCredentialsProviderChain",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
    "spark.kryo.registrator": "is.hail.kryo.HailKryoRegistrator",

    "spark.hadoop.fs.s3.maxConnections": "50000",
    "spark.hadoop.fs.s3.connection.timeout": "120000",
    "spark.hadoop.fs.s3.socket.timeout": "120000",
    "spark.hadoop.fs.s3.maxRetries": "20",
    "spark.hadoop.fs.s3a.threads.max": "256",
    "spark.hadoop.fs.s3a.connection.maximum": "50000",
    "spark.hadoop.fs.s3a.connection.timeout": "120000",
    "spark.hadoop.fs.s3a.socket.timeout": "120000",
    "spark.hadoop.fs.s3a.attempts.maximum": "20",
    "spark.hadoop.fs.s3a.retry.interval": "1000",
    "spark.hadoop.fs.s3a.fast.upload": "true",
    "spark.hadoop.fs.s3a.multipart.size": "104857600",
    "spark.hadoop.fs.s3a.threads.keepalivetime": "60000",
    "spark.hadoop.fs.s3a.connection.establish.timeout": "30000",
    "spark.hadoop.fs.s3a.multipart.purge.age": "86400000"
  },
  "executorCores": 4,
  "executorMemory": "18G",
  "driverCores": 8,
  "driverMemory": "384G"
}

In [2]:
# Initialize Hail against the running SparkContext from Livy/EMR Notebooks
import hail as hl
hl.init(sc, log="/tmp/hail.log")

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1781665374088_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/usr/local/lib/python3.11/site-packages/hail/backend/spark_backend.py:76: UserWarning: Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jar
  warnings.warn(
Running on Apache Spark version 3.5.5-amzn-1
SparkUI available at http://ip-192-168-71-120.ap-southeast-1.compute.internal:40455
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.138-58956ebc28fc
LOGGING: writing to /tmp/hail.log

In [6]:
# source
vds_prefix = "s3://precise-scratch/goypav/SG10K_Health/VDS/"
vds_step3_prefix = vds_prefix + "step3/batch1_1000/"

# input
manifest_uri = "s3://precise-scratch/goypav/SG10K_Health/VDS/vds_step2_manifest_SG10K_Health_batch1_1000.txt"

# output



FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
# -----------------------------
# 1. Read manifest of VDS paths
# -----------------------------
### create vds_step2_manifest.txt
# aws s3 ls s3://precise-scratch/goypav/SG10K_Health/VDS/step2/ \
# | awk '$NF ~ /^batch_.*\.vds\/?$/ {
#     print "s3://precise-scratch/goypav/SG10K_Health/VDS/step2/" $NF
# }' \
# | sort \
# > vds_step2_manifest_SG10K_Health_batch1_1000.txt

# ### copy to s3
# aws s3 cp vds_step2_manifest_SG10K_Health_batch1_1000.txt s3://precise-scratch/goypav/SG10K_Health/VDS/ --dryrun

with hl.current_backend().fs.open(manifest_uri) as f:
    vds_paths = [
        line.strip()
        for line in f
        if line.strip()
    ]

vds_paths = sorted(vds_paths)

print(f"Found {len(vds_paths):,} VDSes")
print("\nFirst 5 VDSes:")
print(*vds_paths[:5], sep="\n")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Found 10 VDSes

First 5 VDSes:
s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0000.vds/
s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0001.vds/
s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0002.vds/
s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0003.vds/
s3://precise-scratch/goypav/SG10K_Health/VDS/step2/batch_0004.vds/

In [8]:
# -----------------------------
# 2. Configure and run combiner
# -----------------------------

import time
from datetime import datetime

out_uri = vds_step3_prefix + "SG10K_Health_batch1_1000.vds"
tmp_uri = vds_step3_prefix + "tmp/"

branch_factor = 10
target_records = 500_000

print(f"Output URI : {out_uri}")
print(f"Temp URI   : {tmp_uri}")
print(f"VDS count  : {len(vds_paths):,}")

start = time.time()

combiner = hl.vds.new_combiner(
    output_path=out_uri,
    temp_path=tmp_uri,
    vds_paths=vds_paths,
    use_genome_default_intervals=True,
    reference_genome="GRCh38",
    branch_factor=branch_factor,
    target_records=target_records,
    gvcf_save_filters=True,
)

print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Starting combiner")

combiner.run()

elapsed = time.time() - start

print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] Combiner completed")
print(f"Runtime: {elapsed:.1f} seconds ({elapsed/60:.2f} minutes)")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output URI : s3://precise-scratch/goypav/SG10K_Health/VDS/step3/batch1_1000/SG10K_Health_batch1_1000.vds
Temp URI   : s3://precise-scratch/goypav/SG10K_Health/VDS/step3/batch1_1000/tmp/
VDS count  : 10
[2026-06-17 03:44:02] Starting combiner
[2026-06-17 04:03:41] Combiner completed
Runtime: 1206.4 seconds (20.11 minutes)

## check output

In [9]:
# -----------------------------
# Read first combined batch VDS
# -----------------------------

vds = hl.vds.read_vds(out_uri)

print(f"Loaded: {out_uri}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Loaded: s3://precise-scratch/goypav/SG10K_Health/VDS/step3/batch1_1000/SG10K_Health_batch1_1000.vds

In [10]:
# -----------------------------
# Inspect schemas
# -----------------------------

print("=== Reference data ===")
vds.reference_data.describe()


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

=== Reference data ===
----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LEN': int32
    'gvcf_qual': float64
    'gvcf_filters': set<str>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [11]:
print("\n=== Variant data ===")
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


=== Variant data ===
----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>


In [12]:
# -----------------------------
# Basic summary
# -----------------------------

print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples():,}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")
print(f"Variant partitions: {vds.variant_data.n_partitions():,}")
print(f"Reference partitions: {vds.reference_data.n_partitions():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 1,000
Total number of variants: 163,820,032
Variant partitions: 5,702
Reference partitions: 5,702